In [0]:
spark.conf.set(
  "fs.azure.account.key.olistdatastoragepradeep.dfs.core.windows.net", 
  "code"
)

In [0]:
path = "abfss://olistdata@olistdatastoragepradeep.dfs.core.windows.net/bronze/olist_customers_dataset.csv"

# 
spark.conf.set(
    "fs.azure.account.key.olistdatastoragepradeep.dfs.core.windows.net",
    "storage-code"
)

# 2) Read CSV
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")   # optional
      .csv(path)
)



In [0]:
storage_account = "olistdatastoragepradeep"
container = "olistdata"

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze/"

orders_path      = base_path + "olist_orders_dataset.csv"
payments_path    = base_path + "olist_order_payments_dataset.csv"
reviews_path     = base_path + "olist_order_reviews_dataset.csv"
items_path       = base_path + "olist_order_items_dataset.csv"
customers_path   = base_path + "olist_customers_dataset.csv"
sellers_path     = base_path + "olist_sellers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
products_path    = base_path + "olist_products_dataset.csv"

orders_df = spark.read.format("csv").option("header", "true").load(orders_path)
payments_df = spark.read.format("csv").option("header", "true").load(payments_path)
reviews_df = spark.read.format("csv").option("header", "true").load(reviews_path)
items_df = spark.read.format("csv").option("header", "true").load(items_path)
customers_df = spark.read.format("csv").option("header", "true").load(customers_path)
sellers_df = spark.read.format("csv").option("header", "true").load(sellers_path)
geolocation_df = spark.read.format("csv").option("header", "true").load(geolocation_path)
products_df = spark.read.format("csv").option("header", "true").load(products_path)

In [0]:
display(orders_df)


Reading data from Pymongo

In [0]:
!pip install pymongo
from pymongo import MongoClient


In [0]:
from pymongo import MongoClient

In [0]:
# importing module
from pymongo import MongoClient
import pandas as pd

hostname = "7uxatx.h.filess.io"
database = "OlistdataNoSQL_fallfront"
port = "27018"
username = "OlistdataNoSQL_fallfront"
password = "d56ac5bf578cf81749052925a5809117636beb1c"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
mydatabase

In [0]:
import pandas as pd
collection = mydatabase['product_categories']

mongo_data = pd.DataFrame(list(collection.find()))

In [0]:
mongo_data

Cleaning the data

In [0]:

from pyspark.sql.functions import col,to_date,datediff,current_date,when

In [0]:
def clean_datafram(df,name):
    print("Cleaning" + name)
    return df.dropDuplicates().na.drop('all')

orders_df = clean_datafram(orders_df,"Orders")
display(orders_df)

In [0]:
# convert Date Colums

orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_delivered_customer_date", to_date(col("order_delivered_customer_date")))\
        .withColumn("order_estimated_delivery_date", to_date(col("order_estimated_delivery_date")))
        

In [0]:
from pyspark.sql.functions import col, datediff

orders_df = (orders_df
    .withColumn("actual_delivery_days",
                datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("estimated_delivery_days",
                datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))
    .withColumn("delay_time",
                col("actual_delivery_days") - col("estimated_delivery_days"))
)

display(orders_df.select(
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "actual_delivery_days",
    "estimated_delivery_days",
    "delay_time"
))

In [0]:
display(orders_df.limit(5))

Joining

In [0]:
(['order_id',
  'customer_id',
  'order_status',
  'order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date',
  'actual_delivery_time',
  'estimated_delivery_time',
  'delay',
  'Delay Time',
  'customer_id',
  'customer_unique_id',
  'customer_zip_code_prefix',
  'customer_city',
  'customer_state',
  'order_id',
  'payment_sequential',
  'payment_type',
  'payment_installments',
  'payment_value'],
 ['order_id',
  'order_item_id',
  'product_id',
  'seller_id',
  'shipping_limit_date',
  'price',
  'freight_value'])

In [0]:
orders_cutomers_df = orders_df.join(customers_df, orders_df.customer_id == customers_df.customer_id,"left")

orders_payments_df = orders_cutomers_df.join(payments_df, orders_cutomers_df.order_id == payments_df.order_id,"left")

orders_items_df = orders_payments_df.join(items_df,"order_id","left")

orders_items_products_df = orders_items_df.join(products_df, orders_items_df.product_id == products_df.product_id,"left")

final_df = orders_items_products_df.join(sellers_df, orders_items_products_df.seller_id == sellers_df.seller_id,"left")

In [0]:
display(final_df)

In [0]:
if '_id' in mongo_data.columns: mongo_data.drop('_id', axis=1, inplace=True)

mongo_sparf_df = spark.createDataFrame(mongo_data)
display(mongo_sparf_df)


In [0]:
final_df= final_df.join(mongo_spark_df,"product_category_name","left")
display(final_df)

In [0]:
def remove_duplicate_columns(df):
    colums = df.columns
    seen_columns =set()
    columns_to_drop= []
    for column in colums:
        if column in seen_columns:
            columns_to_drop.append(column)
        else:
            seen_columns.add(column)

        df_cleaned = df.drop(*columns_to_drop)
    return df_cleaned

final_df = remove_duplicate_columns(final_df)

In [0]:
final_df.write.mode("overwrite").parquet("abfss://olistdata@olistdatastoragepradeep.dfs.core.windows.net/silver")